In [ ]:
!pip install arch
!pip install pytorch-lightning
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import yfinance as yf

from sklearn.preprocessing import StandardScaler

import pytorch_lightning as pl
from pytorch_lightning import Trainer, seed_everything
from pytorch_lightning.loggers import CSVLogger
from torch.utils.data import Dataset, DataLoader
from scipy.stats import norm
from arch.univariate import SkewStudent
from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor
# Check if CUDA is available
print(torch.cuda.is_available())  # Should return True
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from torch.optim.lr_scheduler import ReduceLROnPlateau
import pytorch_lightning as pl
from pytorch_lightning.loggers import CSVLogger
from torch.utils.data import Dataset, DataLoader
from scipy.stats import norm
from arch.univariate import SkewStudent
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 985.1/985.1 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.3/819.3 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 927.3/927.3 kB 54.7 MB/s eta 0:00:00
True


In [ ]:
class TimeseriesDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray, seq_len: int = 1):
        """Initialize TimeseriesDataset
        Args:
            X: Input features array
            y: Target array
            seq_len: Length of sequence for each sample
        """
        if isinstance(X, torch.Tensor):
            self.X = X.float()
        else:
            self.X = torch.tensor(X).float()

        if isinstance(y, torch.Tensor):
            self.y = y.float()
        else:
            self.y = torch.tensor(y).float()

        self.seq_len = seq_len

    def __len__(self):
        return len(self.X) - self.seq_len

    def __getitem__(self, index):
        return (
            self.X[index:index + self.seq_len],
            self.y[index + self.seq_len - 1]
        )

In [ ]:
class ValueAtRiskDataModule(pl.LightningDataModule):
    def __init__(self, df, training_length=1000, seq_len=1, batch_size=128, num_workers=4):
        super().__init__()
        self.df = df
        self.test_case = 0
        self.seq_len = seq_len
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.X_train = None
        self.y_train = None
        self.X_val = None
        self.y_val = None
        self.X_test = None
        self.preprocessing = None
        self.training_length = training_length
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    def prepare_data(self):
        pass

    def setup_train(self):
        # Get data slice for training
        data = self.df.iloc[self.test_case:self.training_length + self.test_case]


        # Split into features and target
        X = data[['log_returns']].values[:self.training_length]
        y = data[['log_returns']].shift(-1).values[:self.training_length]

        # Initialize and fit scaler
        self.preprocessing = StandardScaler()
        self.preprocessing.fit(X)

        # Transform training data
        self.X_train = self.preprocessing.transform(X)
        self.y_train = self.preprocessing.transform(y)

        # Prepare test data
        self.X_test = data[['log_returns']].values[-self.seq_len - 1:-1]
        self.X_test = self.preprocessing.transform(self.X_test)
        self.X_test = torch.tensor(self.X_test, dtype=torch.float32, device=self.device).unsqueeze(0)

        # Convert training data to tensors and move to device
        self.X_train = torch.tensor(self.X_train, dtype=torch.float32)
        self.y_train = torch.tensor(self.y_train, dtype=torch.float32)

    def train_dataloader(self):
        train_dataset = TimeseriesDataset(
            self.X_train,
            self.y_train,
            seq_len=self.seq_len
        )

        return DataLoader(
            train_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=True
        )

    def move_timestep(self):
        self.test_case += 1

    def gather_prediction(self, predictions):
        """
        Store and print predictions.
        Args:
            predictions: Dictionary of predictions or single VaR value in real domain.
        """
        current_position = self.test_case + self.training_length
        current_date = self.df.index[current_position]
        current_return = self.df.iloc[current_position]['log_returns']

        # Extract scalar from Series if necessary
        if isinstance(current_return, pd.Series):
            current_return = current_return.item()
        elif isinstance(current_return, pd.DataFrame):
            # In case multiple values are returned, handle accordingly
            current_return = current_return.values[0][0]

        # Handle dictionary predictions (from skewed model)
        if isinstance(predictions, dict):
            # Initialize columns if needed
            for key in predictions.keys():
                if key not in self.df.columns:
                    self.df[key] = np.nan

            # Store predictions directly without transformation
            for key, value in predictions.items():
                self.df.loc[current_position, key] = value

            # Print status
            print(f"Prediction {self.test_case + 1}")
            print(f"Date: {current_date}")
            print(f"Log Return: {current_return:.6f}")

            for key, value in predictions.items():
                print(f"{key}: {float(value):.6f}")
        else:
            # Handle single VaR prediction
            if 'VaR' not in self.df.columns:
                self.df['VaR'] = np.nan

            # Store prediction directly without transformation
            transformed_prediction = predictions

            self.df.at[current_position, 'VaR'] = transformed_prediction  # Use .at for scalar access

            # Print status
            print(f"Prediction {self.test_case + 1}")
            print(f"Date: {current_date}")
            print(f"Log Return: {current_return}")
            print(f"VaR: {float(transformed_prediction)}")

        print("-" * 50)

In [ ]:
import torch
import numpy as np


def caviar_loss(true, pred, pval=0.025):
    # return torch.mean(-1*((true < var).float() - pval) * (true - var))
    return torch.mean(torch.max(pval*(true - pred), (pval-1)*(true - pred)))


def huber_loss(true, var, pval=torch.tensor(0.025), eps=torch.tensor(0.025)):
    x = true - var
    return torch.mean(torch.cat([
        x[x <= (pval - 1) * eps] * (pval - 1) - 1 / 2 * (pval - 1) ** 2 * eps,
        x[(x > (pval - 1) * eps) & (x <= pval * eps)] ** 2 / (2 * eps),
        x[x > pval * eps] * pval - 1 / 2 * pval ** 2 * eps
    ]))


def garch_normal_loss(true, vol):
    return 1 / 2 * torch.mean(torch.log(vol) + true ** 2 / vol)  # + tf.math.log(2 * tf.constant(np.pi))


def student_loss(true, pred):
    vol = pred[0]
    df = pred[1] + 2

    llh = + 1/2 * (
        torch.log(vol) + (1+df)*torch.log(1 + torch.square(true)/(vol * (df - 2)))
    )

    return llh


def hansen_garch_skewed_student_loss(true, pred):
    vol = pred[:, 0]
    df = pred[:, 2]
    skewness = pred[:, 1]
    true = true[:, 0]

    # Compute constants
    c = torch.lgamma((df + 1) / 2) - torch.lgamma(df / 2) - torch.log(torch.pi * (df - 2)) / 2
    a = 4 * skewness * torch.exp(c) * (df - 2) / (df - 1)
    b = torch.sqrt(1 + 3 * torch.square(skewness) - torch.square(a))

    # Normalize residuals
    z = true / torch.sqrt(vol)

    # Log likelihood components
    lls = torch.log(b) + c - torch.log(vol) / 2
    llf_resid = torch.square((b * z + a) / (1 + torch.sign(z + a / b) * skewness))
    lls -= (df + 1) / 2 * torch.log(1 + llf_resid / (df - 2))

    # Negative log likelihood
    lls *= -1

    return torch.mean(lls)


In [ ]:
class VaRNet(pl.LightningModule):
    def __init__(self,
                 n_features,
                 hidden_size,
                 seq_len,
                 batch_size,
                 num_layers,
                 dropout,
                 learning_rate,
                 criterion,
                 dist):
        super().__init__()
        self.n_features = n_features
        self.hidden_size = hidden_size
        self.seq_len = seq_len
        self.batch_size = batch_size
        self.num_layers = num_layers
        self.dropout = dropout
        self.criterion = criterion
        self.learning_rate = learning_rate
        self.dist = dist

        self.lstm = nn.LSTM(input_size=n_features,
                            hidden_size=hidden_size,
                            num_layers=num_layers,
                            dropout=dropout,
                            batch_first=True)
    def forward(self, x):
        # lstm_out = (batch_size, seq_len, hidden_size)
        lstm_out, _ = self.lstm(x)
        return y_pred


    def training_step(self, batch, batch_idx):
        x, y = batch
        x, y = x.to(self.device), y.to(self.device)
        predictions = self(x)

        # Extract parameters for monitoring
        vol, skew, df = predictions[:, 0], predictions[:, 1], predictions[:, 2]

        # Calculate loss
        loss = self.criterion(y, predictions)

        # Log both loss and parameters
        self.log_dict({
            'train_loss': loss,
            'mean_vol': torch.mean(vol),
            'mean_skew': torch.mean(skew),
            'mean_df': torch.mean(df)
        }, prog_bar=True, on_epoch=True)

        return loss

    def predict_var(self, x):
        """
        Predict VaR and return all parameters
        Returns:
            dict: Dictionary containing VaR and all parameters
        """
        self.eval()
        with torch.no_grad():
            x = x.to(self.device)
            predictions = self(x)
            output = predictions.cpu().numpy()[0]  # Get first prediction

            # Extract and validate parameters
            vol = float(output[0])  # Ensure float type
            skew = float(np.clip(output[1], -0.99, 0.99))  # Skewness
            df = float(np.clip(output[2], 2.0, 300.0))  # Degrees of freedom

            # Calculate VaR using skewed t distribution
            dist = self.dist()
            ppf_value = dist.ppf(0.025, parameters=[df, skew])  # Correct parameter order
            var = np.sqrt(vol) * ppf_value

            # Return dictionary with all parameters
            return {
                'VaR': var,
                'Volatility': vol,
                'Skewness': skew,
                'DegreesOfFreedom': df
            }


class GARCHVaRNet(VaRNet):

    def __init__(self,
                 n_features,
                 hidden_size,
                 seq_len,
                 batch_size,
                 num_layers,
                 dropout,
                 learning_rate,
                 criterion,
                 dist):
        super().__init__(n_features,
                         hidden_size,
                         seq_len,
                         batch_size,
                         num_layers,
                         dropout,
                         learning_rate,
                         criterion,
                         dist)

        self.softplus = torch.nn.Softplus()

    def forward(self, x):
        # lstm_out = (batch_size, seq_len, hidden_size)
        lstm_out, _ = self.lstm(x)
        y_pred = self.linear(lstm_out[:, -1])
        y_pred = self.linear2(y_pred)
        y_pred = self.linear3(y_pred)
        return self.softplus(y_pred)

class SkewedGARCHVaRNet(VaRNet):
    def __init__(self,
                 n_features,
                 hidden_size,
                 seq_len,
                 batch_size,
                 num_layers,
                 dropout,
                 learning_rate,
                 criterion,
                 dist):
        super().__init__(n_features,
                         hidden_size,
                         seq_len,
                         batch_size,
                         num_layers,
                         dropout,
                         learning_rate,
                         criterion,
                         dist)

        # Save hyperparameters
        self.save_hyperparameters()
        self.linear =nn.Linear(self.hidden_size, 64)
        self.linear2 = nn.Linear(64, 32)



        # Output heads
        # Volatility head: 32 -> 1
        self.linear3_1 = nn.Sequential(
            nn.Linear(32, 1)
        )
        # Skewness head: 32 -> 1
        self.linear3_2 = nn.Sequential(
            nn.Linear(32, 1)
        )

        # Degrees of freedom head: 32 -> 1
        self.linear3_3 = nn.Sequential(
            nn.Linear(32, 1)
        )

        # Activation functions
        self.softplus = nn.Softplus()
        self.tanh = nn.Tanh()
        self.relu = nn.ReLU()

 # Initialize parameters
        self.init_parameters()

    def init_parameters(self):
        """
        Initialize parameters for skewness and degrees of freedom using He Initialization.
        """
        with torch.no_grad():
            # Initialize degrees of freedom layer to output ~10
            final_df_layer = self.linear3_3[-1]

            # He (Kaiming) Normal Initialization for weights
            nn.init.kaiming_normal_(final_df_layer.weight, a=0, mode='fan_in', nonlinearity='relu')

            # Initialize biases to a constant value
            nn.init.constant_(final_df_layer.bias, 20.0)  # Ensure ReLU + bias outputs ~10


    def forward(self, x):
        """
        Forward pass through the network
        Args:
            x: Input tensor of shape (batch_size, seq_len, n_features)
        Returns:
            Concatenated tensor of [volatility, skewness, degrees_of_freedom]
        """
        # Process through LSTM
        lstm_out, _ = self.lstm(x)
        last_lstm = lstm_out[:, -1]  # Take last LSTM output

        # Feed-forward processing
        y_pred = self.linear(last_lstm)
        y_pred = self.linear2(y_pred)

        # Output heads with activations
        vol = self.softplus(self.linear3_1(y_pred)) + 1e-8  # Ensure positive volatility
        skew = self.tanh(self.linear3_2(y_pred))            # Constrain skew to [-1, 1]
        df = self.relu(self.linear3_3(y_pred)) + 3     # Ensure df > 3

        # Combine outputs
        return torch.cat([vol, skew, df], dim=1)


In [ ]:
def experiment_nikkei(model_name='garch_skew'):
    # Set up device and print info
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # Model setup with proper configuration
    if model_name == 'garch_skew':
        model_class = SkewedGARCHVaRNet
        dist = SkewStudent
        loss = hansen_garch_skewed_student_loss
    elif model_name == 'garch_norm':
        model_class = GARCHVaRNet
        dist = norm
        loss = garch_normal_loss
    elif model_name == 'caviar':
        model_class = VaRNet
        dist = None
        loss = caviar_loss
    elif model_name == 'caviar_huber':
        model_class = VaRNet
        dist = None
        loss = huber_loss
    else:
        raise ValueError(f"Unknown model name: {model_name}")

    # Download and prepare Nikkei data
    nikkei = yf.download('^N225', start='2000-01-01')
    data = nikkei[['Close']].copy()
    data['log_returns'] = np.log(data['Close'] / data['Close'].shift(1))
    data = data.dropna()

    # Initialize columns based on model type
    data['VaR'] = np.nan
    if model_name == 'garch_skew':
        data['Volatility'] = np.nan
        data['Skewness'] = np.nan
        data['DegreesOfFreedom'] = np.nan


    # Training parameters setup
    memory_sizes = [10]
    total_samples = len(data)
    sample_points = [int(total_samples * 0.73)]
    sample_dates = [data.index[point].strftime('%Y-%m-%d') for point in sample_points]

    class ModelWithScheduler(model_class):
        def configure_optimizers(self):
            optimizer = torch.optim.AdamW(
                self.parameters(),
                lr=self.learning_rate,
                weight_decay=0.01
            )

            # Example: CosineAnnealingWarmRestarts
            scheduler = {
                'scheduler': torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
                    optimizer,
                    T_0=2,        # first restart epoch
                    T_mult=2,      # each restart interval doubles
                    eta_min=1e-10
                ),
                'interval': 'epoch',
                'frequency': 1
            }

            return [optimizer], [scheduler]

    # Training loop
    for mem_size in memory_sizes:
        # Training parameters
        p = dict(
            training_length=1000,
            seq_len=mem_size,
            batch_size=128,
            criterion=loss,
            max_epochs=150,
            n_features=1,
            hidden_size=100,
            num_layers=1,
            dropout=0.0,
            learning_rate=3e-4,
            num_train=5
        )

        for sample_start in sample_dates:
            # Prepare data for current sample
            current_data = data.loc[(data.index > sample_start)].copy()

            # Initialize data module with appropriate columns
            if model_name == 'garch_skew':
                columns = ['log_returns', 'VaR', 'Volatility', 'Skewness', 'DegreesOfFreedom']
            else:
                columns = ['log_returns', 'VaR']

            dm = ValueAtRiskDataModule(
                df=current_data[columns].copy(),
                training_length=p['training_length'],
                seq_len=p['seq_len'],
                batch_size=p['batch_size'],
                num_workers=4
            )

            for test_case in range(p['num_train']):
                print(f"Processing test case {test_case + 1}/{p['num_train']}")

                # Initialize and configure model with scheduler
                model = ModelWithScheduler(
                    n_features=p['n_features'],
                    hidden_size=p['hidden_size'],
                    seq_len=p['seq_len'],
                    batch_size=p['batch_size'],
                    criterion=p['criterion'],
                    num_layers=p['num_layers'],
                    dropout=p['dropout'],
                    learning_rate=p['learning_rate'],
                    dist=dist
                ).to(device)
                # Configure trainer with callbacks
                trainer = Trainer(
                    max_epochs=p['max_epochs'],
                    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
                    devices=1,
                    enable_progress_bar=True,
                    log_every_n_steps=1,
                    enable_model_summary=True,
                    enable_checkpointing=False,
                    callbacks=[early_stopping, lr_monitor]
                )

                # Setup and train
                dm.setup_train()
                trainer.fit(model, dm)
                # Make predictions
                try:
                    with torch.no_grad():
                        predictions = model.predict_var(dm.X_test)

                        if isinstance(predictions, dict):
                            # Get current position information
                            current_position = dm.test_case + dm.training_length
                            current_date = dm.df.index[current_position]
                            current_return = dm.df.iloc[current_position]['log_returns']

                            # Print current date and return
                            print(f"\nDate: {current_date}")
                            print(f"Log Return: {float(current_return.iloc[0]):.10f}")

                            # Print scaled domain predictions
                            print("\nPredictions (scaled domain):")
                            print(f"  Volatility (scaled):       {predictions['Volatility']:.10f}")
                            print(f"  Skewness:                  {predictions['Skewness']:.10f}")
                            print(f"  DegreesOfFreedom:          {predictions['DegreesOfFreedom']:.10f}")
                            print(f"  VaR (scaled):              {predictions['VaR']:.10f}")

                            # Transform predictions to real domain
                            transformed_predictions = predictions.copy()

                            # Handle volatility properly: sqrt -> inverse_transform -> square
                            scaled_std = np.sqrt(predictions['Volatility'])
                            original_std = dm.preprocessing.inverse_transform([[scaled_std]])[0][0]
                            transformed_predictions['Volatility'] = original_std ** 2

                            # Handle VaR: direct inverse transform
                            transformed_predictions['VaR'] = dm.preprocessing.inverse_transform([[predictions['VaR']]])[0][0]

                            # Keep skewness and df as is
                            transformed_predictions['Skewness'] = predictions['Skewness']
                            transformed_predictions['DegreesOfFreedom'] = predictions['DegreesOfFreedom']

                            # Print real domain predictions
                            print("\nPredictions (real domain):")
                            print(f"  Volatility (real):         {transformed_predictions['Volatility']:.10f}")
                            print(f"  Skewness:                  {transformed_predictions['Skewness']:.10f}")
                            print(f"  DegreesOfFreedom:          {transformed_predictions['DegreesOfFreedom']:.10f}")
                            print(f"  VaR (real):                {transformed_predictions['VaR']:.10f}")

                            # Store predictions using gather_prediction
                            dm.gather_prediction(transformed_predictions)

                        else:
                            # Handle single VaR prediction
                            if isinstance(predictions, torch.Tensor):
                                predictions = predictions.cpu().numpy()
                            scaled_predictions = np.array(predictions).reshape(-1, 1)

                            # Inverse transform the VaR prediction
                            real_predictions = dm.preprocessing.inverse_transform(scaled_predictions)
                            dm.gather_prediction(float(real_predictions[0][0]))

                except Exception as e:
                    print(f"Error during prediction: {str(e)}")
                    continue
                # Move to next timestep
                dm.move_timestep()



            # Save results for current sample
            output_filename = f'nikkei_{model_name}_{sample_start}_{mem_size}.csv'
            dm.df[columns].to_csv(output_filename)
            print(f"Results saved to {output_filename}")

if __name__ == "__main__":
    experiment_nikkei('garch_skew')

[*********************100%***********************]  1 of 1 completed

Using device: cuda



INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name      | Type       | Params | Mode 
-------------------------------------------------
0 | lstm      | LSTM       | 41.2 K | train
1 | linear    | Linear     | 6.5 K  | train
2 | linear2   | Linear     | 2.1 K  | train
3 | linear3_1 | Sequential | 33     | train
4 | linear3_2 | Sequential | 33     | train
5 | linear3_3 | Sequential | 33     | train
6 | softplus  | Softplus   | 0      | train
7 | tanh      | Tanh       | 0      | train
8 | relu      | ReLU       | 0      | train
-------------------------------------------------
49.8 K    Trainable params
0         Non-trainable params
49.8 K    Total params
0.

Processing test case 1/5


Training: |          | 0/? [00:00<?, ?it/s]

...

Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.callbacks.early_stopping:Metric train_loss improved. New best score: 1.422
INFO:pytorch_lightning.callbacks.early_stopping:Metric train_loss improved by 0.008 >= min_delta = 0.0. New best score: 1.414
INFO:pytorch_lightning.callbacks.early_stopping:Metric train_loss improved by 0.007 >= min_delta = 0.0. New best score: 1.407
INFO:pytorch_lightning.callbacks.early_stopping:Metric train_loss improved by 0.008 >= min_delta = 0.0. New best score: 1.399
INFO:pytorch_lightning.callbacks.early_stopping:Metric train_loss improved by 0.006 >= min_delta = 0.0. New best score: 1.393
INFO:pytorch_lightning.callbacks.early_stopping:Metric train_loss improved by 0.003 >= min_delta = 0.0. New best score: 1.390
INFO:pytorch_lightning.callbacks.early_stopping:Metric train_loss improved by 0.007 >= min_delta = 0.0. New best score: 1.383
INFO:pytorch_lightning.callbacks.early_stopping:Metric train_loss improved by 0.013 >= min_delta = 0.0. New best score: 1.370
INFO:pytorch_lightni


Date: 2022-05-10 00:00:00
Log Return: -0.0058011427

Predictions (scaled domain):
  Volatility (scaled):       0.7427771091
  Skewness:                  -0.0928013250
  DegreesOfFreedom:          3.9035100937
  VaR (scaled):              -1.7823811385

Predictions (real domain):
  Volatility (real):         0.0001225052
  Skewness:                  -0.0928013250
  DegreesOfFreedom:          3.9035100937
  VaR (real):                -0.0222434450
Prediction 2
Date: 2022-05-10 00:00:00
Log Return: -0.005801
VaR: -0.022243
Volatility: 0.000123
Skewness: -0.092801
DegreesOfFreedom: 3.903510
--------------------------------------------------
Processing test case 3/5


Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.callbacks.early_stopping:Metric train_loss improved. New best score: 1.426
INFO:pytorch_lightning.callbacks.early_stopping:Metric train_loss improved by 0.007 >= min_delta = 0.0. New best score: 1.419
INFO:pytorch_lightning.callbacks.early_stopping:Metric train_loss improved by 0.006 >= min_delta = 0.0. New best score: 1.413
INFO:pytorch_lightning.callbacks.early_stopping:Metric train_loss improved by 0.008 >= min_delta = 0.0. New best score: 1.405
INFO:pytorch_lightning.callbacks.early_stopping:Metric train_loss improved by 0.006 >= min_delta = 0.0. New best score: 1.399
INFO:pytorch_lightning.callbacks.early_stopping:Metric train_loss improved by 0.003 >= min_delta = 0.0. New best score: 1.397
INFO:pytorch_lightning.callbacks.early_stopping:Metric train_loss improved by 0.007 >= min_delta = 0.0. New best score: 1.390
INFO:pytorch_lightning.callbacks.early_stopping:Metric train_loss improved by 0.013 >= min_delta = 0.0. New best score: 1.376
INFO:pytorch_lightni